# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you in loading, exploring, and analyzing the `FAIR^2` dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s and explore sample records.

In [ ]:
# List available record sets by @id and their fields
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets detected in the metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet name: {rs.name}\n  @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (type: {field.data_type}, @id: {field.id})")
        print()

In [ ]:
# Show sample records (first 2) from the primary record set
# Pick the first found record set for demonstration
if record_sets:
    record_set_id = record_sets[0].id
    print(f"First 2 records of RecordSet with @id: {record_set_id}")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i == 1:
            break

## 3. Data Extraction
Load data from selected record set(s) into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(df.head(2))
    print('-'*60)
# For demonstration, pick the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping. This example uses a numeric field and a group/categorical field (by their `@id`).

In [ ]:
import numpy as np
# Choose a numeric field
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Get all numeric-like columns
    numeric_fields = []
    for field in dataset.metadata.record_sets[0].fields:
        # Types may be: 'Float', 'Integer', 'Number', etc.
        if str(field.data_type).lower() in ["float", "integer", "number"]:
            if field.id in df.columns:
                numeric_fields.append(field.id)

    if not numeric_fields:
        print("No numeric fields detected for EDA.")
    else:
        numeric_field = numeric_fields[0]

        # Remove missing or invalid values that could break analysis
        # Try to convert if necessary
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        threshold = np.nanmean(df[numeric_field])
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        print(filtered_df[[numeric_field]].head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt grouping by a non-numeric field (e.g., first field which is not numeric)
        group_field = None
        for field in dataset.metadata.record_sets[0].fields:
            if str(field.data_type).lower() not in ["float", "integer", "number"] and field.id in df.columns:
                group_field = field.id
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (showing mean of numeric columns):")
            print(grouped_df.head())
        else:
            print("No suitable group/categorical field found for grouping.")

## 5. Visualization
Visualize numeric field distributions and relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If there is a group field, show boxplot
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_record_set_id])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have loaded the FAIR^2 mass spectrometry dataset, previewed the available record sets, fields, and explored example data. Both statistical and visual analysis can be applied using the flexible, standards-based Croissant metadata. For deeper analysis, reference field `@id`s in transforming or modeling the data, ensuring reproducibility and clarity.